# 02 — Run the pipeline on the local sample reports

Ingests whatever report files currently sit in `data/raw`, parses them, runs the classifier, and prints probabilities alongside the evidence sentence that drove each label — for manually eyeballing predictions on the small local sample set.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import pandas as pd

from report2label.ingestion import read_report_file
from report2label.parsing.report_parser import parse_report
from report2label.extraction.label_mapper import LabelVocabulary
from report2label.extraction.predictor import LabelPredictor
from report2label.extraction.thresholding import resolve_thresholds
from report2label.models.model_loader import ModelConfig
from report2label.utils.io import list_report_files, read_yaml

pipeline_config = read_yaml(PROJECT_ROOT / "configs" / "pipeline.yaml")
model_config = ModelConfig.from_yaml(PROJECT_ROOT / "configs" / "model.yaml")
label_vocab = LabelVocabulary.from_yaml(PROJECT_ROOT / "configs" / "labels.yaml")
thresholds = resolve_thresholds(
    label_vocab.names, model_config.default_threshold, pipeline_config.get("label_thresholds")
)
predictor = LabelPredictor(model_config, label_vocab, thresholds, pipeline_config["evidence"])

In [ ]:
raw_dir = PROJECT_ROOT / pipeline_config["paths"]["raw_reports_dir"]
files = list_report_files(raw_dir, pipeline_config["ingestion"]["supported_extensions"])
print(f"Found {len(files)} report(s) under {raw_dir}")

parsed_reports = []
for file_path in files:
    raw_text = read_report_file(file_path)
    parsed = parse_report(
        raw_text,
        section_headers=pipeline_config["sections"]["headers"],
        classifier_input_sections=pipeline_config["sections"]["classifier_input_sections"],
        phi_removal_enabled=pipeline_config["phi_removal"]["enabled"],
        source_path=str(file_path),
    )
    parsed_reports.append(parsed)
    print(f"{file_path.name}: ct_id={parsed.ct_id}")

In [ ]:
predictions = [
    predictor.predict(p.classifier_text, ct_id=p.ct_id, source_path=p.source_path)
    for p in parsed_reports
]

summary = pd.DataFrame(
    [{"report": Path(p.source_path).name, **pred.probabilities} for p, pred in zip(parsed_reports, predictions)]
)
summary.set_index("report")

In [ ]:
# Inspect evidence for whatever labels a given report predicted positive.
report_index = 0
prediction = predictions[report_index]
positive_labels = [name for name, value in prediction.labels.items() if value == 1]
print(f"{Path(prediction.source_path).name} — positive labels: {positive_labels}")

for label in positive_labels:
    print(f"\n[{label}] p={prediction.probabilities[label]:.3f}")
    for item in (prediction.evidence or {}).get(label, []):
        print(f"    ({item['probability']:.3f}) {item['sentence']}")